# PiSA-SR · GIRAFFE

На GPU ещё не проверен.

Запускайте сверху вниз в отдельной сессии Kaggle с GPU и Internet. Зависимости устанавливаются в отдельное Python-окружение.

Подключите датасет через Add Input и укажите его путь в разделе 4. Структура: `<корень>/<sample_id>/degraded.png`; `hint.png` нужен только для просмотра.

[Официальный репозиторий](https://github.com/csslc/PiSA-SR) · [Зафиксированная версия](https://github.com/csslc/PiSA-SR/tree/9c76e12e261d1980148738a0cff41e5e27d18c9f)

## 1. Репозиторий


In [ ]:
from pathlib import Path
import datetime
import hashlib
import json
import os
import shutil
import struct
import subprocess
import sys
import uuid
import zipfile

REPO_URL = 'https://github.com/csslc/PiSA-SR.git'
COMMIT = '9c76e12e261d1980148738a0cff41e5e27d18c9f'
WORK_ROOT = Path('/kaggle/working/giraffe_baselines') if Path('/kaggle/working').exists() else Path.cwd() / '.baseline_work'
WORK_ROOT = WORK_ROOT.resolve()
REPO_DIR = WORK_ROOT / 'external' / 'pisa_sr'

def run(*args, cwd=REPO_DIR, env=None):
    subprocess.run([str(a) for a in args], cwd=cwd, env=env, check=True)

REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    run('git', 'clone', REPO_URL, REPO_DIR, cwd=WORK_ROOT)
origin = subprocess.check_output(['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
if origin.removesuffix('.git') != REPO_URL.removesuffix('.git'):
    raise RuntimeError(f'В {REPO_DIR} находится другой репозиторий: {origin}')
# В upstream отслеживается __pycache__, который Python перезаписывает при импорте.
changed = subprocess.check_output(['git', '-C', str(REPO_DIR), 'diff', 'HEAD', '--name-only'], text=True).splitlines()
changed = [name for name in changed if not ('__pycache__' in Path(name).parts and name.endswith('.pyc'))]
if changed:
    raise RuntimeError('В baseline-репозитории изменены отслеживаемые файлы. Сохраните изменения отдельно.')
run('git', 'fetch', 'origin', COMMIT)
run('git', 'checkout', '--detach', COMMIT)
actual = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if actual != COMMIT:
    raise RuntimeError(f'Версия репозитория не совпала: {actual}')
print(f'pisa_sr: {actual}')


## 2. Окружение

PiSA-SR, увеличение ×4, fp16, пустой промпт. Используется официальный `test_pisasr.py`. Он принимает seed, но не применяет его, поэтому seed устанавливается перед запуском. RAM в этом режиме не используется.


In [ ]:
# Зависимости метода устанавливаются отдельно от ядра Jupyter.
if sys.platform != 'linux' or not shutil.which('nvidia-smi'):
    raise RuntimeError('Нужна Linux-сессия с NVIDIA GPU. В Kaggle включите GPU и Internet. На Mac CUDA недоступна.')
run('nvidia-smi', cwd=WORK_ROOT)
run(sys.executable, '-m', 'pip', 'install', 'uv==0.9.7', cwd=WORK_ROOT)
ENV_DIR = WORK_ROOT / 'envs' / 'pisa_sr'
if not (ENV_DIR / 'bin/python').exists():
    run(sys.executable, '-m', 'uv', 'venv', '--python', '3.10', '--seed', ENV_DIR, cwd=WORK_ROOT)
PYTHON = ENV_DIR / 'bin/python'

def run_python(source, *args, env=None):
    run(PYTHON, '-c', source, *args, env=env)

run(PYTHON, '-m', 'pip', 'install', 'pip==24.3.1', 'setuptools==75.6.0', 'wheel==0.45.1')
run(PYTHON, '-m', 'pip', 'install', 'torch==2.0.1', 'torchvision==0.15.2', '--index-url', 'https://download.pytorch.org/whl/cu118')
run(PYTHON, '-m', 'pip', 'install', '-r', 'requirements.txt', 'accelerate==0.28.0', 'gdown==5.2.0')
run_python("import torch; assert torch.cuda.is_available(), 'PyTorch не видит CUDA'; print(torch.__version__, torch.cuda.get_device_name(0))")
run(PYTHON, '-m', 'pip', 'check')


## 3. Веса

`pisa_sr.pkl` скачивается с Google Drive, SD2.1 Base — из публичной копии `sd2-community` на Hugging Face с закреплённой версией. Можно указать другую копию через `GIRAFFE_SD2_BASE`. При ограничении Google Drive скачайте файл вручную по ссылке из сообщения об ошибке.


In [ ]:
run_python(r"""from pathlib import Path
import json, hashlib, os
from huggingface_hub import snapshot_download, model_info
import gdown
assets = []


def hf_snapshot(repo, destination, patterns, revision=None):
    destination = Path(destination)
    revision = revision or model_info(repo).sha
    snapshot_download(repo_id=repo, revision=revision, local_dir=destination,
                      allow_patterns=patterns, ignore_patterns=['*.onnx*', '*.msgpack', '*.fp16.*', '*.pyc'])
    assets.append({'source': f'https://huggingface.co/{repo}', 'revision': revision,
                   'local': str(destination), 'patterns': patterns})
    return destination

def drive_file(folder_url, filename, destination):
    destination = Path(destination)
    if not destination.is_file() or destination.stat().st_size == 0:
        entries = gdown.download_folder(url=folder_url, skip_download=True, quiet=True)
        matches = [entry for entry in (entries or []) if Path(entry.path).name == filename]
        if len(matches) != 1:
            raise RuntimeError(f'Не найден однозначный {filename}. Скачайте его вручную из {folder_url} и положите в {destination}; затем повторите ячейку.')
        destination.parent.mkdir(parents=True, exist_ok=True)
        part = destination.with_name(destination.name + '.part')
        if not gdown.download(id=matches[0].id, output=str(part), quiet=False, resume=True):
            raise RuntimeError(f'Google Drive не отдал {filename}. Загрузите файл вручную в {destination}.')
        part.replace(destination)
    with destination.open('rb') as f:
        digest = hashlib.file_digest(f, 'sha256').hexdigest() if hasattr(hashlib, 'file_digest') else None
    if digest is None:
        h = hashlib.sha256()
        with destination.open('rb') as f:
            for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
                h.update(chunk)
        digest = h.hexdigest()
    assets.append({'source': folder_url, 'file': filename, 'local': str(destination),
                   'bytes': destination.stat().st_size, 'sha256': digest})
    return destination

# Публичная копия SD2.1 Base: исходный репозиторий Stability AI недоступен.
# Версия зеркала закреплена; GIRAFFE_SD2_BASE позволяет указать другую копию.
base = os.environ.get('GIRAFFE_SD2_BASE', 'sd2-community/stable-diffusion-2-1-base')
base_revision = '4e63672c03103b6c636b8fb4119ba982469b2955' if base == 'sd2-community/stable-diffusion-2-1-base' else None
if Path(base).is_dir():
    base_dir = Path(base).resolve()
    assets.append({'source': 'user-provided local SD2.1-base', 'local': str(base_dir)})
else:
    base_dir = hf_snapshot(base, 'preset/models/stable-diffusion-2-1-base',
        ['model_index.json', 'scheduler/*', 'tokenizer/*', 'text_encoder/*.json',
         'text_encoder/*.safetensors', 'unet/*.json', 'unet/*.safetensors',
         'vae/*.json', 'vae/*.safetensors'], revision=base_revision)
for folder in ['scheduler', 'tokenizer', 'text_encoder', 'unet', 'vae']:
    if not (base_dir / folder).is_dir():
        raise RuntimeError(f'Неполная базовая модель: {base_dir / folder}')

drive_file('https://drive.google.com/drive/folders/1oLetijWNd59xwJE5oU-eXylQBifxWdss',
           'pisa_sr.pkl', 'preset/models/pisa_sr.pkl')
# В test_pisasr.py промпт пустой, RAM в этом пути инференса не вызывается.
Path('giraffe_assets.json').write_text(json.dumps({'assets': assets, 'base_model': str(base_dir)}, indent=2))
""")


## 4. Параметры и данные

Вход модели — только `degraded.png`. Каждое выполнение создаёт новую папку запуска.


In [ ]:
DATASET_ROOT = Path('/kaggle/input/datasets/markunyaa/giraffe-dataset/giraffe_dataset')
RESULTS_ROOT = WORK_ROOT / 'results'
SEED = 42
SETTINGS = {
    'upscale': 4,
    'process_size': 512,
    'default': True,
    'align_method': 'adain',
    'mixed_precision': 'fp16',
    'vae_decoder_tiled_size': 64,
    'vae_encoder_tiled_size': 1024,
    # 64 латента = 512 пикселей: плитка не больше минимального входа process_size.
    'latent_tiled_size': 64,
    'latent_tiled_overlap': 32,
    'initial_prompt': '',
}

DATASET_ROOT = DATASET_ROOT.expanduser().resolve()
RESULTS_ROOT = RESULTS_ROOT.expanduser().resolve()
sources = sorted(DATASET_ROOT.glob('*/degraded.png'))
if not sources:
    raise FileNotFoundError(f'Не найдены */degraded.png в {DATASET_ROOT}. Укажите путь из Kaggle Add Input.')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RUN_DIR = WORK_ROOT / 'runs' / 'pisa_sr' / RUN_ID
INPUT_DIR = RUN_DIR / 'input'
RAW_DIR = RUN_DIR / 'raw'
INPUT_DIR.mkdir(parents=True)
RAW_DIR.mkdir()
SAMPLES = {}
for src in sources:
    sample_id = src.parent.name
    target = INPUT_DIR / f'{sample_id}.png'
    shutil.copy2(src, target)
    SAMPLES[sample_id] = src
ASSETS = json.loads((REPO_DIR / 'giraffe_assets.json').read_text())
RUN_CONFIG = {'method': 'pisa_sr', 'repo': REPO_URL, 'commit': COMMIT, 'seed': SEED,
              'settings': SETTINGS, 'weights': ASSETS, 'dataset_root': str(DATASET_ROOT),
              'input_dir': str(INPUT_DIR), 'raw_dir': str(RAW_DIR), 'run_id': RUN_ID}
CONFIG_PATH = RUN_DIR / 'run.json'
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False))
INFERENCE_OK = False
EXPORT_OK = False
print(f'{len(SAMPLES)} примеров: ' + ', '.join(SAMPLES))
print(f'Результаты: {RESULTS_ROOT}')


## 5. Инференс

Для повторного запуска сначала выполните ячейку параметров.


In [ ]:
INFERENCE_OK = False
EXPORT_OK = False
if any(RAW_DIR.iterdir()):
    raise RuntimeError('Для повторного инференса заново выполните ячейку параметров: она создаст новую папку запуска.')

# Официальный parser принимает seed, но main его не применяет: фиксируем RNG перед runpy.
seeded_entrypoint = r"""import sys, runpy, random, numpy as np, torch
s=int(sys.argv.pop(1))
random.seed(s)
np.random.seed(s)
torch.manual_seed(s)
torch.cuda.manual_seed_all(s)
sys.argv[0]='test_pisasr.py'
runpy.run_path('test_pisasr.py', run_name='__main__')
"""
COMMAND = [str(PYTHON), '-c', seeded_entrypoint, str(SEED),
    '--pretrained_model_path', ASSETS['base_model'], '--pretrained_path', 'preset/models/pisa_sr.pkl',
    '--input_image', str(INPUT_DIR), '--output_dir', str(RAW_DIR), '--seed', str(SEED)]
for key, value in SETTINGS.items():
    if key == 'initial_prompt':
        continue
    if isinstance(value, bool):
        if value:
            COMMAND.append('--' + key)
    else: COMMAND += ['--' + key, str(value)]
run(*COMMAND)
PREDICTIONS = {sid: RAW_DIR / f'{sid}.png' for sid in SAMPLES}
INFERENCE_OK = True


## 6. Сохранение

Результаты: `results/<sample_id>/pisa_sr.png`. Рядом сохраняются параметры, версии весов, seed и размеры изображений. Архив содержит результаты и сведения о запуске. Повторный экспорт заменяет PNG и JSON этого метода; архивы сохраняются отдельно.


In [ ]:
EXPORT_OK = False
if not INFERENCE_OK:
    raise RuntimeError('Сначала выполните инференс текущего запуска без ошибок.')
missing = [str(p) for p in PREDICTIONS.values() if not p.is_file()]
if missing:
    raise RuntimeError('Метод не создал все результаты: ' + ', '.join(missing))

def png_size(path):
    with Path(path).open('rb') as f:
        header = f.read(24)
    if header[:8] != b'\x89PNG\r\n\x1a\n':
        raise ValueError(f'Ожидался PNG: {path}')
    return list(struct.unpack('>II', header[16:24]))

for path in PREDICTIONS.values():
    png_size(path)
freeze = subprocess.check_output([str(PYTHON), '-m', 'pip', 'freeze'], text=True)
(RUN_DIR / 'environment.txt').write_text(freeze)
RUN_CONFIG['command'] = COMMAND
RUN_CONFIG['status'] = 'completed'
RUN_CONFIG['completed_at_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False))
EXPORTED = []
for sid, predicted in PREDICTIONS.items():
    out_dir = RESULTS_ROOT / sid
    out_dir.mkdir(parents=True, exist_ok=True)
    output = out_dir / 'pisa_sr.png'
    shutil.copy2(predicted, output)
    metadata = dict(RUN_CONFIG, sample_id=sid,
        input_size=png_size(INPUT_DIR / f'{sid}.png'), output_size=png_size(output),
        input_sha256=hashlib.sha256((INPUT_DIR / f'{sid}.png').read_bytes()).hexdigest(),
        output_sha256=hashlib.sha256(output.read_bytes()).hexdigest())
    info = out_dir / 'pisa_sr.json'
    info.write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
    EXPORTED.extend([output, info])
archive = WORK_ROOT / f'giraffe_pisa_sr_{RUN_ID}.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in EXPORTED:
        z.write(path, Path('results') / path.relative_to(RESULTS_ROOT))
    z.write(CONFIG_PATH, 'run.json')
    z.write(RUN_DIR / 'environment.txt', 'environment.txt')
EXPORT_OK = True
print(f'Сохранено {len(PREDICTIONS)} результатов: {archive}')


## 7. Просмотр

`hint.png` — референс для сравнения. Он не поступает в модель и может отличаться ракурсом или кадрированием.


In [ ]:
if not EXPORT_OK:
    raise RuntimeError('Сначала завершите сохранение результатов текущего запуска.')
from IPython.display import display, HTML, FileLink
import base64
import html
for sid, source in SAMPLES.items():
    items = [('Вход: degraded.png', source), ('Результат: pisa_sr', RESULTS_ROOT / sid / 'pisa_sr.png')]
    hint = source.parent / 'hint.png'
    if hint.is_file():
        items.append(('Референс (hint.png)', hint))
    panels = []
    for label, path in items:
        encoded = base64.b64encode(path.read_bytes()).decode('ascii')
        panels.append(f'<div style="max-width:32%;"><p>{html.escape(label)}</p><img style="width:100%;height:auto" src="data:image/png;base64,{encoded}"></div>')
    display(HTML('<h3>' + html.escape(sid) + '</h3><div style="display:flex;gap:12px;align-items:flex-start">' + ''.join(panels) + '</div>'))
# Относительная ссылка обслуживается Jupyter; в Kaggle архив также виден в Output.
display(FileLink(os.path.relpath(archive, Path.cwd())))
